# 23-16 · Отмена и конфликт при восстановлении

Практика к разделу [«Отмена последней операции»](../../site/chapters/glava-23/23-16-otmena-operacii.html). Использует настоящий пакет `safesort`.

## Цель

Применить план, отменить его настоящей `safesort.manifest.undo()` и убедиться, что при конфликте (на исходном месте уже что-то есть) отмена отказывается перезаписывать.

## Рабочий пример

In [1]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan
from safesort.planner import build_plan
from safesort.executor import apply_plan
from safesort.manifest import write_manifest, read_manifest, find_latest_manifest, undo

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

(koren / "otchet.pdf").write_text("отчёт", encoding="utf-8")
(koren / "zametka.txt").write_text("заметка", encoding="utf-8")

nastrojki = Config()
fajly = scan(koren, nastrojki)
plan = build_plan(fajly, koren, nastrojki)
rezultaty = apply_plan(plan)
_manifest_obj, put_k_manifestu = write_manifest(koren, rezultaty)

print("Перемещено файлов:", sum(1 for r in rezultaty if r.completed))
print("Манифест записан в:", put_k_manifestu)

Перемещено файлов: 2
Манифест записан в: /tmp/tmpc91y5u7u/.safesort/history/20260824T020028546288.json


## Проверка результата — файлы действительно перемещены

In [2]:
assert not (koren / "otchet.pdf").exists()
assert not (koren / "zametka.txt").exists()
assert (koren / "Sorted" / "documents" / "otchet.pdf").exists()
assert put_k_manifestu.exists()
print("Верно: оба файла оказались в Sorted/documents/, манифест записан на диск.")

Верно: оба файла оказались в Sorted/documents/, манифест записан на диск.


## Эксперимент — undo восстанавливает файлы

In [3]:
najdennyj_manifest = find_latest_manifest(koren)
manifest_dlya_otmeny = read_manifest(najdennyj_manifest)

rezultat_otmeny = undo(manifest_dlya_otmeny)

assert (koren / "otchet.pdf").exists()
assert (koren / "zametka.txt").exists()
assert rezultat_otmeny.conflicts == ()
print("Верно: оба файла вернулись на исходное место, конфликтов не было.")

Верно: оба файла вернулись на исходное место, конфликтов не было.


## Задание ★★ Самостоятельная задача — конфликт при повторной отмене

Повторите перемещение, затем создайте новый файл на исходном месте ДО отмены — и проверьте, что undo отказывается его затирать.

In [4]:
fajly2 = scan(koren, nastrojki)
plan2 = build_plan(fajly2, koren, nastrojki)
rezultaty2 = apply_plan(plan2)
_manifest_obj2, put_k_manifestu2 = write_manifest(koren, rezultaty2)

# кто-то создал новый файл на месте, откуда только что уехал otchet.pdf
(koren / "otchet.pdf").write_text("новый файл, положенный руками", encoding="utf-8")

manifest_dlya_otmeny2 = read_manifest(find_latest_manifest(koren))
rezultat_otmeny2 = undo(manifest_dlya_otmeny2)

assert len(rezultat_otmeny2.conflicts) == 1
assert rezultat_otmeny2.conflicts[0].source == koren / "otchet.pdf"
assert (koren / "otchet.pdf").read_text(encoding="utf-8") == "новый файл, положенный руками"
print("Верно: undo отказался перезаписать новый файл и сообщил о конфликте.")

tmpdir.cleanup()

Refusing to undo /tmp/tmpc91y5u7u/Sorted/documents/otchet.pdf -> /tmp/tmpc91y5u7u/otchet.pdf: a file already exists at the original location: /tmp/tmpc91y5u7u/otchet.pdf


Верно: undo отказался перезаписать новый файл и сообщил о конфликте.
